# LAC News Monitor

In [1]:
import os, json, re
import feedparser
import pandas as pd
from google import genai
from google.genai import types
from arcgis.gis import GIS
from arcgis.features import FeatureSet
import urllib

In [ ]:
#ArcGIS Login Details
AGOL_ORG_URL = "https://idb-gis.maps.arcgis.com"
AGOL_CLIENT_ID = "xunIRcqrmgb4gB0T"
AGOL_CLIENT_SECRET = "c927d97d667c40c0b7f07350943b475e"
AGOL_LAYER_ID_2 = "dfbb6dbc3a384d8aa436803b896eacd6"
GEMINI_API_KEY ="AQ.Ab8RN6KmWv1DMLQI5CYmxF1ltErJK8j3gbeaQtT75LPuRmquCw"

In [3]:
import requests

def resolve_url(google_url):
    """Follows Google RSS redirection links to extract the final publisher URL."""
    try:
        # Use stream=True so Python only reads response headers without downloading the full webpage
        resp = requests.get(google_url, allow_redirects=True, timeout=5, stream=True)
        return resp.url
    except Exception:
        # Fall back to truncated original if the request times out
        return google_url[:250]

In [4]:
# =========================================================
# Step 1: Build Multi-Query RSS URLs & Pre-Filter Noise
# =========================================================

# 1A. English & Spanish targeted queries (keeps terms under Google's 30-term limit)
query_en = (
    '(disaster OR volcano OR earthquake OR flood OR landslide OR storm OR wildfire OR drought) '
    '("Latin America" OR Caribbean OR Guatemala OR Mexico OR Colombia OR Peru OR Ecuador OR Chile OR Brazil) '
    'when:3d'
)

query_es = (
    '(volcan OR erupcion OR sismo OR terremoto OR inundacion OR deslave OR derrumbe OR incendio) '
    '(Guatemala OR Mexico OR Colombia OR Peru OR Ecuador OR Chile OR "America Latina") '
    'when:3d'
)

url_en = f"https://news.google.com/rss/search?q={urllib.parse.quote(query_en)}&hl=en-US&gl=US&ceid=US:en"
url_es = f"https://news.google.com/rss/search?q={urllib.parse.quote(query_es)}&hl=es-419&gl=MX&ceid=MX:es"

# 1B. Fetch and combine entries from both feeds
raw_entries = []
for url in [url_en, url_es]:
    feed = feedparser.parse(url)
    raw_entries.extend(feed.entries)

# 1C. Filter US noise and remove duplicate links
us_blacklist = [r"\bUS\b", r"\bU\.S\.\b", r"\bUnited States\b", r"\bFlorida\b", r"\bTexas\b", r"\bCalifornia\b", r"\bFEMA\b"]
seen_links = set()
candidates = []

for entry in raw_entries:
    raw_link = entry.get('link', '')
    title = entry.get('title', '')
    summary = entry.get('summary', '')
    combined_text = f"{title} {summary}"

    if raw_link in seen_links or any(re.search(pat, combined_text, re.IGNORECASE) for pat in us_blacklist):
        continue

    seen_links.add(raw_link)
    
    # Resolve to actual publisher URL (e.g., Prensa Libre, Reuters, BBC)
    clean_url = resolve_url(raw_link)

    candidates.append({
        "id": len(candidates),
        "title": title,
        "summary": summary,
        "published": entry.get('published', ''),
        "link": clean_url
    })

print(f"Retrieved {len(candidates)} unique candidates across EN and ES feeds.")

# =========================================================
# Step 2: Single Gemini Batch Processing Call
# =========================================================
client = genai.Client(api_key=GEMINI_API_KEY)

prompt = f"""
You are an expert GIS data analyst tracking natural hazards in Latin America and the Caribbean (LAC).
Review the following JSON list of candidate news items ({len(candidates)} items total).

Tasks:
1. Filter out any items that are NOT active/recent natural disasters occurring within Latin America or the Caribbean.
2. Extract standard values for each valid disaster matching the required JSON schema.
3. Return ONLY a JSON array of valid objects.

Candidate News Items:
{json.dumps(candidates, indent=2)}

Target JSON Array Output Format:
[
  {{
    "country": "Primary English country name (e.g., Guatemala, Colombia, Brazil)",
    "disaster_type": "Drought/Earthquake/Flooding/Extreme Cold/Extreme Heat/Cyclones/Landslides/Storms/Tsunami/Wildfires/Tornado/Volcanic Eruptions",
    "headline_en": "Translated headline in English",
    "summary_en": "Clean English summary without HTML tags",
    "date_reported": "YYYY-MM-DD HH:MM:SS",
    "source_url": "Original article URL"
  }}
]
"""

response = client.models.generate_content(
    model='gemini-3.5-flash',
    contents=prompt,
    config=types.GenerateContentConfig(response_mime_type="application/json")
)

parsed_articles = json.loads(response.text)
print(f"Gemini verified {len(parsed_articles)} active LAC disaster records.")



Retrieved 199 unique candidates across EN and ES feeds.
Gemini verified 6 active LAC disaster records.


In [ ]:
df = pd.DataFrame(parsed_articles)

if not df.empty:
    # Convert string dates to native pandas datetime objects for AGOL compatibility
    if "date_reported" in df.columns:
        df["date_reported"] = pd.to_datetime(df["date_reported"], errors='coerce')

    df.to_csv("disasters_lac.csv", index=False)

    # Connect to AGOL and select layer
    gis = GIS(AGOL_ORG_URL, client_id=AGOL_CLIENT_ID, client_secret=AGOL_CLIENT_SECRET)
    item = gis.content.get(AGOL_LAYER_ID_2)
    layer = item.layers[0] if item.layers else item.tables[0]

    # Delete existing rows
    layer.delete_features(where="1=1")

    # Push new features
    feature_set = FeatureSet.from_dataframe(df)
    edit_response = layer.edit_features(adds=feature_set)

    # Validate response status directly from AGOL
    add_results = edit_response.get('addResults', [])
    successes = [r for r in add_results if r.get('success')]
    failures = [r for r in add_results if not r.get('success')]

    print(f"AGOL Response: {len(successes)} rows added successfully | {len(failures)} failures.")

    if failures:
        print("AGOL Error Detail:", failures[0].get('error'))
else:
    print("Warning: No valid disaster articles found. AGOL layer was not updated.")

AGOL Response: 6 rows added successfully | 0 failures.
